# `PSCarlemanBasis`

Pivot-switching Carleman. Use it when Carleman's truncation error is too large. The states are the
same as `CarlemanBasis`, and only the truncation differs: a monomial above the degree becomes its
Taylor polynomial about a reference state instead of disappearing.

Results below are checked against Eqs. (3) and (16) of
[arXiv:2411.16233](https://arxiv.org/abs/2411.16233). For switching and accuracy over time, see
[`psc_logistic_switching`](../../examples/psc_logistic_switching.ipynb).

## 1. The basis functions

Identical to Carleman:

$$y_{\boldsymbol\alpha}(u)=\prod_i u_i^{\alpha_i},\qquad|\boldsymbol\alpha|\le P.$$

The degree is written $P$ rather than $K$ by convention, following arXiv:2411.16233.

Nothing about the states changes. The labels, the ordering and the indexing are `CarlemanBasis`.

They are monomials, so this basis is not square-integrable either. `Liouville` and
`KoopmanVonNeumann` raise `IncompatibleBasisError` on it, exactly as on `CarlemanBasis`
([`03_linearization`](../03_linearization.ipynb)).

## 2. The state class: `CarlemanState`

The same class Carleman uses. The states are monomials either way, so `PSCarlemanBasis` inherits the
state type rather than defining its own.

In [1]:
import sympy as sp

from lin_ode.basis import (CarlemanBasis, CarlemanKRule, CarlemanURule,
                           PSCarlemanBasis)

s = sp.Symbol("s")  # one pivot shared by every site

carleman, psc = CarlemanBasis(1, 3), PSCarlemanBasis(1, 3, pivot=s)

print("Carleman basis[(0, 2)] =", carleman[(0, 2)])
print("PSC      basis[(0, 2)] =", psc[(0, 2)])
print("same state type:", type(carleman[(0, 2)]) is type(psc[(0, 2)]))

Carleman basis[(0, 2)] = CarlemanState((0, 2))
PSC      basis[(0, 2)] = CarlemanState((0, 2))
same state type: True


## 3. The basis class: `PSCarlemanBasis`

`PSCarlemanBasis(n_vars, degree, pivot)` takes one argument more than Carleman.

`pivot` is a number, a symbol shared by every site, or an `sp.IndexedBase` read as `pivot[site]` for
a per-site pivot. Keep it symbolic and substitute later to get several pivots out of one assembly.

In [2]:
shared = PSCarlemanBasis(3, 2, sp.Symbol("s"))
per_site = PSCarlemanBasis(3, 2, sp.IndexedBase("s"))

for basis in (CarlemanBasis(3, 2), shared, per_site):
    print(f"{type(basis).__name__:20} len={len(basis):3}")

print("\nshared pivot  , fold of u_0^3:", sp.expand(shared.truncation_rule((0, 3))))
print("per-site pivot, fold of u_0^3:", sp.expand(per_site.truncation_rule((0, 3))))

CarlemanBasis        len= 10
PSCarlemanBasis      len= 10
PSCarlemanBasis      len= 10

shared pivot  , fold of u_0^3: s**3*CarlemanState() - 3*s**2*CarlemanState((0, 1)) + 3*s*CarlemanState((0, 2))
per-site pivot, fold of u_0^3: CarlemanState()*s[0]**3 - 3*s[0]**2*CarlemanState((0, 1)) + 3*s[0]*CarlemanState((0, 2))


## 4. Indexing

Unchanged from Carleman. The states are the same, so the flat ordering and both index directions
behave identically.

In [3]:
basis = PSCarlemanBasis(1, 3, pivot=s)
states = [basis.structured_idx_of_flat_index(k) for k in range(len(basis))]

print(f"{len(basis)} states:", [f"y{sum(p for _, p in a)}" for a in states])
print("as structured indices:", states)

4 states: ['y0', 'y1', 'y2', 'y3']
as structured indices: [(), ((0, 1),), ((0, 2),), ((0, 3),)]


## 5. The computation rules

:::{note}
Sections 5 to 7 are how the library produces a value, not something you call.
`OperatorMatrix` and `Vector` apply these rules for you. Read them when you are adding a
basis of your own, or when an element is not the number you expected.
:::

`PSCarlemanBasis` subclasses `CarlemanBasis` and inherits its rules unchanged, so the two sections
say the same thing. A linearized operator needs two things from a basis: what multiplication by
$u_s$ does to a basis state, and what $\partial/\partial u_s$ does.

On monomials both are one term:

$$U_s\,y_{\boldsymbol\alpha}=y_{\boldsymbol\alpha+\mathbf e_s},
\qquad
K_s\,y_{\boldsymbol\alpha}=\alpha_s\,y_{\boldsymbol\alpha-\mathbf e_s}.$$

$U$ raises the power at site $s$. $K$ lowers it and leaves the old power behind as a factor. Both
are index arithmetic, which is why an element costs the same at any grid size.

The generator $\mathcal K=\sum_s f_s(u)\,K_s$ then gives

$$\mathcal K\,y_{\boldsymbol\alpha}=\sum_s \alpha_s\,f_s(u)\,y_{\boldsymbol\alpha-\mathbf e_s},$$

a sum of polynomials. The coefficient of a target state $y_m$ is the coefficient of that monomial.

In [4]:
print("U on (site 0, n=2):", CarlemanURule()(sp.Integer(0), sp.Integer(2)))
print("K on (site 0, n=2):", CarlemanKRule()(sp.Integer(0), sp.Integer(2)))

print("\nsame rule classes as CarlemanBasis:",
      issubclass(PSCarlemanBasis, CarlemanBasis))

U on (site 0, n=2): CarlemanState((0, 3))
K on (site 0, n=2): 2*CarlemanState((0, 1))

same rule classes as CarlemanBasis: True


The rules are identical, so nothing so far distinguishes PSC. The difference appears one step later.
$U$ raises a monomial above the degree, and the fold decides what replaces it.

## 6. Truncation

Above the degree, a monomial is replaced by its degree-$P$ Taylor polynomial about the pivot
(arXiv:2411.16233, Eqs. 9 and 11). On one site,

$$u^{P+1}\;\longmapsto\;u^{P+1}-(u-s)^{P+1},$$

so the discarded part is the remainder $(u-s)^{P+1}$. The $u^{P+1}$ cancels the leading term of
that expansion, so what survives is its low-order part.

Moving the pivot as the state moves is the *switching* in the name.

A general monomial spreads over every sub-monomial:

$$y_m\;\longmapsto\;\sum_{\beta\le m,\;|\beta|\le P}T_{\beta m}(s)\,y_\beta$$

$$T_{\beta m}(s)=\Bigl(\prod_i s_i^{\,m_i-\beta_i}\Bigr)
\sum_{\substack{\beta\,\le\,j\,\le\,m\\ |j|\,\le\,P}}\ \prod_i(-1)^{\,j_i-\beta_i}
\binom{m_i}{j_i}\binom{j_i}{\beta_i}$$

$\beta\le m$ is componentwise; $|\beta|$ is the total degree.

It follows from substituting $u_i=s_i+\varepsilon_i$, keeping $|j|\le P$, and re-expanding
$\varepsilon_i^{j_i}=(u_i-s_i)^{j_i}$. The powers of $s$ combine to $s_i^{m_i-\beta_i}$ independently
of $j$, so only the $j$-sum depends on $P$.

In [5]:
from sympy import binomial


def fold_matrix(out_of_basis: list[int], degree: int, pivot: sp.Symbol) -> sp.Matrix:
    """T[beta, m] on one site: rows beta = 0..degree, one column per m."""

    def entry(beta: int, m: int) -> sp.Expr:
        prefactor = pivot ** (m - beta)
        total = sum(
            (-1) ** (j - beta) * binomial(m, j) * binomial(j, beta)
            for j in range(beta, min(m, degree) + 1)
        )
        return sp.expand(prefactor * total)

    return sp.Matrix(degree + 1, len(out_of_basis), lambda b, c: entry(b, out_of_basis[c]))


columns = [4, 5, 6]
T = fold_matrix(columns, psc.degree, s)

print(f"T[beta, m] for P = {psc.degree}, m = {columns}   (rows beta = 0..{psc.degree})")
display(T)

# every entry must equal the coefficient truncation_rule actually produces
for col, m in enumerate(columns):
    folded = sp.expand(psc.truncation_rule((0, m)))
    for beta in range(psc.degree + 1):
        target = psc.state_type((0, beta)) if beta else psc.state_type()
        assert sp.simplify(sp.expand(folded.coeff(target)) - T[beta, col]) == 0, (beta, m)
print("\nevery entry matches truncation_rule")

T[beta, m] for P = 3, m = [4, 5, 6]   (rows beta = 0..3)


Matrix([
[  -s**4,  -4*s**5, -10*s**6],
[ 4*s**3,  15*s**4,  36*s**5],
[-6*s**2, -20*s**3, -45*s**4],
[    4*s,  10*s**2,  20*s**3]])


every entry matches truncation_rule


Column $m=4$ is the $|m|=P+1$ case, and its entries are the coefficients of $u^4-(u-s)^4$.

That is the fold: the part of $u^4$ that the basis cannot hold, re-expressed about the pivot.

In [6]:
print("in basis, returned unchanged:")
display(psc.truncation_rule((0, 2)))

print("out of basis -- the degree-3 Taylor polynomial about s:")
display(sp.expand(psc.truncation_rule((0, 4))))

print("conventional truncation drops it:")
display(carleman.truncation_rule((0, 4)))

# what PSC discards is the Taylor remainder, O((x - s)^(P+1))
u, eps = sp.symbols("u epsilon")
approx = sp.expand(psc.truncation_rule((0, 4))).replace(
    psc.state_type, lambda *factors: sp.Mul(*[u**power for _, power in factors])
)
print(f"discarded remainder of y4: {sp.factor(u**4 - approx)}"
      f"  =  {sp.expand((u**4 - approx).subs(u, s + eps))} at u = s + eps")

in basis, returned unchanged:


CarlemanState((0, 2))

out of basis -- the degree-3 Taylor polynomial about s:


-s**4*CarlemanState() + 4*s**3*CarlemanState((0, 1)) - 6*s**2*CarlemanState((0, 2)) + 4*s*CarlemanState((0, 3))

conventional truncation drops it:


0

discarded remainder of y4: (s - u)**4  =  epsilon**4 at u = s + eps


## 7. Expressions after truncation

Here the rule does real work, so `truncate_expression` is more than `sp.expand`. A monomial above
the degree comes back as a combination of in-basis ones, with pivot-dependent coefficients.

In [7]:
x = sp.IndexedBase("x")
scalar = PSCarlemanBasis(1, 3, pivot=s)

print("PSC      x_0**4 ->", sp.expand(scalar.truncate_expression(x[0] ** 4, x)))
print("Carleman x_0**4 ->", CarlemanBasis(1, 3).truncate_expression(x[0] ** 4, x))

PSC      x_0**4 -> -s**4 + 4*s**3*x[0] - 6*s**2*x[0]**2 + 4*s*x[0]**3
Carleman x_0**4 -> x[0]**4


The fold reaches every sub-monomial, not just the one above the degree. On more than one site a
state of degree $P+1$ comes back as a combination over the whole box $\beta\le m$:

In [8]:
two_site = PSCarlemanBasis(2, 2, pivot=sp.IndexedBase("s"))
y = sp.IndexedBase("y")

print("degree 3 on two sites, folded onto the degree-2 basis:")
display(sp.expand(two_site.truncate_expression(y[0] ** 2 * y[1], y)))

degree 3 on two sites, folded onto the degree-2 basis:


s[0]**2*s[1] - s[0]**2*y[1] - 2*s[0]*s[1]*y[0] + 2*s[0]*y[0]*y[1] + s[1]*y[0]**2

The remainder is $O\bigl((x-s)^{P+1}\bigr)$, so the folded expression agrees with the original to
order $P$ about the pivot. Expanding both at $x=s+\varepsilon$ shows where they part company:

In [9]:
eps = sp.Symbol("epsilon")
folded = sp.expand(scalar.truncate_expression(x[0] ** 4, x))

for label, expr in (("x^4      ", x[0] ** 4), ("folded   ", folded)):
    print(label, sp.expand(expr.subs(x[0], s + eps)))
print("difference", sp.factor(sp.expand((x[0] ** 4 - folded).subs(x[0], s + eps))))

x^4       epsilon**4 + 4*epsilon**3*s + 6*epsilon**2*s**2 + 4*epsilon*s**3 + s**4
folded    4*epsilon**3*s + 6*epsilon**2*s**2 + 4*epsilon*s**3 + s**4
difference epsilon**4


The two agree through $\varepsilon^{3}$ and differ at $\varepsilon^{4}=\varepsilon^{P+1}$.

So the fold is exact to the truncation order, and its error is the first discarded Taylor term.

Two notes before using this on a grid.

| note | consequence |
|---|---|
| a per-site pivot needs the folding site pinned to a concrete index | a free index makes the fold a sum over the grid, returned as an unevaluated `sp.Sum`. Call `.doit()` on the element to expand it site by site |
| truncation is the full Taylor polynomial, not the leading term alone | a quadratic right-hand side is unaffected; a cubic one diverges (see [`psc_fig4_phase_field`](../../examples/psc_fig4_phase_field.ipynb)) |

---

[← CarlemanBasis](carleman.ipynb) · [Notebook index](../../README.md) · [GaussianHermiteBasis →](gaussian_hermite.ipynb)